In [1]:
import argparse
import os
import random,numpy,pandas
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

In [2]:
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data
import torchvision
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.utils as vutils
import torchvision.transforms.functional as RF
from PIL import Image
import numpy as np
import random,cv2,pandas,os,numpy
import matplotlib.pyplot as plt
import matplotlib.animation as animation

In [3]:
seed = 999
print("Random Seed: ", seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

nz = 100
beta1 = 0.5
lr = 0.0001
batch_size=100000
ngpu=1
ngf,nc = 3,3
ndf = 64

device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")

Random Seed:  999


In [4]:
x = pandas.read_csv("/kaggle/input/rohlik-sales-forecasting-challenge-v2/sales_train.csv").sort_values(by=['date', 'warehouse'])

x = torch.from_numpy(numpy.nan_to_num(x[['total_orders', 'sell_price_main', 'type_0_discount', 'type_1_discount',
                                         'type_2_discount', 'type_3_discount', 'type_4_discount',  'type_5_discount', 
                                         'type_6_discount']].to_numpy().astype(numpy.float32)).reshape((13, 308263, 9)))
x.shape #4315682

class RF_NET(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.rafire = nn.Sequential(       
            nn.Linear(9, 1524),
            nn.Linear(1524, 824),
            nn.Linear(824, 424),
            nn.Linear(424, 124),
            nn.Linear(124, 100)
        )

    def forward(self,x):
        
        return self.rafire(x)
rf_net = RF_NET().type(torch.float32).to(device).eval()
rf_net= nn.DataParallel(rf_net).to(device)
#rf_net.load_state_dict(torch.load("/kaggle/input/encoder-weight-data/weight.pth",weights_only=False,map_location=torch.device('cpu')))


torch.save(rf_net.state_dict(),f'/kaggle/working/weight.pth')


"""j=0
for i in x:
    print(f"loding batch : {j}")
    if j==0:
        encode = rf_net(i).cpu().detach().numpy().reshape((len(i),10,10))
        print(encode.shape)
    else:
        encode = numpy.concatenate((encode, rf_net(i).cpu().detach().numpy().reshape((len(i),10,10))), axis=0)
    #if j==25:
    #    break
    j+=1

x=torch.Tensor(encode)"""

'j=0\nfor i in x:\n    print(f"loding batch : {j}")\n    if j==0:\n        encode = rf_net(i).cpu().detach().numpy().reshape((len(i),10,10))\n        print(encode.shape)\n    else:\n        encode = numpy.concatenate((encode, rf_net(i).cpu().detach().numpy().reshape((len(i),10,10))), axis=0)\n    #if j==25:\n    #    break\n    j+=1\n\nx=torch.Tensor(encode)'

In [5]:
x.shape

torch.Size([13, 308263, 9])